# Algorithm Comparison & Evaluation

## Overview

This notebook compares the performance of three collaborative filtering algorithms:

1. **Item-Based CF**: Memory-based, cosine similarity
2. **User-Based CF**: Memory-based, Pearson correlation with mean-centering
3. **SVD Matrix Factorization**: Model-based, latent factor learning

**Evaluation Criteria**:
- **Accuracy**: RMSE, MAE
- **Speed**: Training time, prediction time
- **Coverage**: Percentage of users/items that can be recommended
- **Scalability**: Memory usage, computational complexity
- **Interpretability**: How easy to explain recommendations

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 2. Load Results from All Three Algorithms

In [ ]:
# Load results CSV files generated by each notebook
try:
    item_based_results = pd.read_csv('item_based_cf_results.csv')
    print("✓ Item-Based CF results loaded")
except:
    print("✗ Item-Based CF results not found. Run 1_item_based_cf.ipynb first.")
    item_based_results = None

try:
    user_based_results = pd.read_csv('user_based_cf_results.csv')
    print("✓ User-Based CF results loaded")
except:
    print("✗ User-Based CF results not found. Run 2_user_based_cf.ipynb first.")
    user_based_results = None

try:
    svd_results = pd.read_csv('svd_results.csv')
    print("✓ SVD results loaded")
except:
    print("✗ SVD results not found. Run 3_svd_matrix_factorization.ipynb first.")
    svd_results = None

In [ ]:
# Combine all results into single DataFrame
results_list = []

if item_based_results is not None:
    results_list.append(item_based_results)

if user_based_results is not None:
    results_list.append(user_based_results)

if svd_results is not None:
    results_list.append(svd_results)

if len(results_list) == 0:
    print("ERROR: No results found. Please run the algorithm notebooks first.")
else:
    all_results = pd.concat(results_list, ignore_index=True)
    print(f"\nCombined results from {len(results_list)} algorithms")
    print("\nAll Results:")
    print(all_results)

## 3. Performance Comparison Table

In [ ]:
# Create comparison table
comparison_df = all_results[['algorithm', 'rmse', 'mae', 'coverage', 
                             'training_time_minutes', 'prediction_time_ms']].copy()

# Round for better display
comparison_df['rmse'] = comparison_df['rmse'].round(4)
comparison_df['mae'] = comparison_df['mae'].round(4)
comparison_df['coverage'] = comparison_df['coverage'].round(2)
comparison_df['training_time_minutes'] = comparison_df['training_time_minutes'].round(2)
comparison_df['prediction_time_ms'] = comparison_df['prediction_time_ms'].round(2)

# Sort by RMSE (best first)
comparison_df = comparison_df.sort_values('rmse')

print("="*80)
print("ALGORITHM COMPARISON TABLE")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Identify best algorithm
best_rmse_algo = comparison_df.iloc[0]['algorithm']
best_rmse = comparison_df.iloc[0]['rmse']

print(f"\n🏆 BEST ALGORITHM (by RMSE): {best_rmse_algo} with RMSE = {best_rmse:.4f}")

## 4. Visualizations

### 4.1 Accuracy Metrics Comparison

In [ ]:
# Plot RMSE and MAE side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RMSE comparison
axes[0].bar(comparison_df['algorithm'], comparison_df['rmse'], color=['steelblue', 'coral', 'seagreen'])
axes[0].set_ylabel('RMSE', fontsize=12)
axes[0].set_title('Root Mean Squared Error (Lower is Better)', fontsize=14, fontweight='bold')
axes[0].set_ylim([0, comparison_df['rmse'].max() * 1.2])
axes[0].grid(axis='y', alpha=0.3)

# Add values on bars
for i, v in enumerate(comparison_df['rmse']):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

# MAE comparison
axes[1].bar(comparison_df['algorithm'], comparison_df['mae'], color=['steelblue', 'coral', 'seagreen'])
axes[1].set_ylabel('MAE', fontsize=12)
axes[1].set_title('Mean Absolute Error (Lower is Better)', fontsize=14, fontweight='bold')
axes[1].set_ylim([0, comparison_df['mae'].max() * 1.2])
axes[1].grid(axis='y', alpha=0.3)

# Add values on bars
for i, v in enumerate(comparison_df['mae']):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('comparison_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()

print("Accuracy comparison plot saved as: comparison_accuracy.png")

### 4.2 Speed Comparison

In [ ]:
# Plot training time and prediction time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training time
axes[0].bar(comparison_df['algorithm'], comparison_df['training_time_minutes'], 
           color=['steelblue', 'coral', 'seagreen'])
axes[0].set_ylabel('Time (minutes)', fontsize=12)
axes[0].set_title('Training Time (Lower is Faster)', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

for i, v in enumerate(comparison_df['training_time_minutes']):
    axes[0].text(i, v + 1, f'{v:.1f}m', ha='center', fontweight='bold')

# Prediction time
axes[1].bar(comparison_df['algorithm'], comparison_df['prediction_time_ms'], 
           color=['steelblue', 'coral', 'seagreen'])
axes[1].set_ylabel('Time (ms per rating)', fontsize=12)
axes[1].set_title('Prediction Time (Lower is Faster)', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

for i, v in enumerate(comparison_df['prediction_time_ms']):
    axes[1].text(i, v + 0.5, f'{v:.2f}ms', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('comparison_speed.png', dpi=300, bbox_inches='tight')
plt.show()

print("Speed comparison plot saved as: comparison_speed.png")

### 4.3 Coverage Comparison

In [ ]:
# Plot coverage
plt.figure(figsize=(10, 6))
bars = plt.bar(comparison_df['algorithm'], comparison_df['coverage'], 
               color=['steelblue', 'coral', 'seagreen'])
plt.ylabel('Coverage (%)', fontsize=12)
plt.title('Algorithm Coverage (Higher is Better)', fontsize=14, fontweight='bold')
plt.ylim([0, 110])
plt.axhline(y=90, color='red', linestyle='--', alpha=0.5, label='Target: 90%')
plt.grid(axis='y', alpha=0.3)
plt.legend()

for i, v in enumerate(comparison_df['coverage']):
    plt.text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('comparison_coverage.png', dpi=300, bbox_inches='tight')
plt.show()

print("Coverage comparison plot saved as: comparison_coverage.png")

### 4.4 Radar Chart: Multi-Dimensional Comparison

In [ ]:
from math import pi

# Normalize metrics for radar chart (higher is better for all)
# Accuracy: 1 / RMSE (so higher is better)
# Speed: 1 / time (so higher is faster)
# Coverage: as-is (already percentage)

radar_data = comparison_df.copy()
radar_data['accuracy_score'] = 1 / radar_data['rmse'] * 10  # Scale for visibility
radar_data['speed_score'] = 1 / radar_data['training_time_minutes'] * 100
radar_data['prediction_speed'] = 1 / radar_data['prediction_time_ms'] * 100
radar_data['coverage_score'] = radar_data['coverage']

# Normalize all scores to 0-100 scale
for col in ['accuracy_score', 'speed_score', 'prediction_speed', 'coverage_score']:
    max_val = radar_data[col].max()
    radar_data[col] = (radar_data[col] / max_val) * 100

# Setup radar chart
categories = ['Accuracy', 'Training Speed', 'Prediction Speed', 'Coverage']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

colors = ['steelblue', 'coral', 'seagreen']

for idx, row in radar_data.iterrows():
    values = [row['accuracy_score'], row['speed_score'], 
              row['prediction_speed'], row['coverage_score']]
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2, label=row['algorithm'], color=colors[idx])
    ax.fill(angles, values, alpha=0.15, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=12)
ax.set_ylim(0, 100)
ax.set_title('Algorithm Comparison (Radar Chart)\nHigher is Better', 
             size=16, fontweight='bold', y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True)

plt.tight_layout()
plt.savefig('comparison_radar.png', dpi=300, bbox_inches='tight')
plt.show()

print("Radar chart saved as: comparison_radar.png")

## 5. Detailed Analysis & Recommendations

### 5.1 Algorithm Strengths & Weaknesses

In [ ]:
print("="*80)
print("ALGORITHM ANALYSIS")
print("="*80)

print("\n1. ITEM-BASED COLLABORATIVE FILTERING")
print("-" * 80)
print("Strengths:")
print("  ✓ More stable than user-based (item relationships don't change often)")
print("  ✓ Good for cold-start users (new users with few ratings)")
print("  ✓ Interpretable (can explain why: 'You rated Movie X, similar to Movie Y')")
print("  ✓ Computationally efficient (fewer items than users typically)")

print("\nWeaknesses:")
print("  ✗ Requires pre-computing item similarities (memory intensive)")
print("  ✗ Cold-start for new items (no ratings to compute similarity)")
print("  ✗ Accuracy lower than model-based methods")

print("\nBest Use Cases:")
print("  • E-commerce (Amazon-style recommendations)")
print("  • Content platforms with stable catalog")
print("  • When interpretability is important")

print("\n" + "="*80)
print("\n2. USER-BASED COLLABORATIVE FILTERING")
print("-" * 80)
print("Strengths:")
print("  ✓ Intuitive and easy to explain ('People like you also liked...')")
print("  ✓ Can discover new genres/items from similar users")
print("  ✓ Social aspect (recommendations feel personalized)")

print("\nWeaknesses:")
print("  ✗ Less stable (user preferences change over time)")
print("  ✗ Scalability issues (more users than items typically)")
print("  ✗ Requires mean-centering (users rate on different scales)")
print("  ✗ Cold-start for new users")

print("\nBest Use Cases:")
print("  • Social platforms (find similar users)")
print("  • Niche communities with stable user base")
print("  • When social proof is important")

print("\n" + "="*80)
print("\n3. SVD MATRIX FACTORIZATION")
print("-" * 80)
print("Strengths:")
print("  ✓ Best accuracy (lowest RMSE/MAE)")
print("  ✓ Fast predictions (just dot product)")
print("  ✓ Scalable to large datasets")
print("  ✓ Learns latent patterns automatically")
print("  ✓ 100% coverage (can predict for any user-item pair)")
print("  ✓ Handles sparsity well")

print("\nWeaknesses:")
print("  ✗ Black box (hard to explain why a movie was recommended)")
print("  ✗ Requires hyperparameter tuning")
print("  ✗ Training slower than memory-based methods")
print("  ✗ Cold-start still an issue (no training data for new users/items)")

print("\nBest Use Cases:")
print("  • Production systems (Netflix, Spotify)")
print("  • Large-scale applications")
print("  • When accuracy is priority")
print("  • Real-time predictions needed")

### 5.2 When to Use Each Algorithm

In [ ]:
print("="*80)
print("DECISION GUIDE: WHICH ALGORITHM TO USE?")
print("="*80)

print("\n🎯 Choose ITEM-BASED CF if:")
print("  1. You need interpretable recommendations")
print("  2. Your item catalog is relatively stable")
print("  3. You have cold-start users (new users joining)")
print("  4. Memory for storing similarity matrix is available")
print("  5. Real-time updates not critical")

print("\n🎯 Choose USER-BASED CF if:")
print("  1. You want social-proof style recommendations")
print("  2. Your user base is small to medium (< 100K users)")
print("  3. User preferences are stable over time")
print("  4. You can afford to recompute similarities periodically")
print("  5. Discovering new genres/items is important")

print("\n🎯 Choose SVD if:")
print("  1. Accuracy is your top priority")
print("  2. You have large-scale data (millions of ratings)")
print("  3. You need fast real-time predictions")
print("  4. You can afford training time and hyperparameter tuning")
print("  5. Interpretability is not critical")

print("\n🎯 Hybrid Approach (Recommended for Production):")
print("  • Use SVD for accuracy and speed")
print("  • Use Item-Based CF for new users (cold-start)")
print("  • Use content-based filtering for new items")
print("  • Blend multiple approaches for robustness")

## 6. Performance Benchmarks

In [ ]:
# Compare against research benchmarks
print("="*80)
print("PERFORMANCE BENCHMARKS")
print("="*80)

print("\nTarget Performance (from analysis.md):")
print("  RMSE < 0.90 (good)")
print("  RMSE < 0.85 (excellent, competitive with research)")
print("  MAE < 0.70 (good)")
print("  Coverage > 90% (production-ready)")

print("\nYour Results:")
for idx, row in comparison_df.iterrows():
    rmse_status = "✓ EXCELLENT" if row['rmse'] < 0.85 else ("✓ GOOD" if row['rmse'] < 0.90 else "⚠ NEEDS IMPROVEMENT")
    mae_status = "✓ GOOD" if row['mae'] < 0.70 else "⚠ NEEDS IMPROVEMENT"
    coverage_status = "✓ EXCELLENT" if row['coverage'] > 90 else "⚠ MODERATE"
    
    print(f"\n{row['algorithm']}:")
    print(f"  RMSE: {row['rmse']:.4f} - {rmse_status}")
    print(f"  MAE:  {row['mae']:.4f} - {mae_status}")
    print(f"  Coverage: {row['coverage']:.1f}% - {coverage_status}")

## 7. Improvement Opportunities

In [ ]:
print("="*80)
print("IMPROVEMENT OPPORTUNITIES")
print("="*80)

print("\n1. HYBRID RECOMMENDER SYSTEM")
print("-" * 80)
print("Combine collaborative filtering + content-based filtering:")
print("  • Use movie metadata (genres, actors, directors, keywords)")
print("  • Weighted combination: 0.7 × CF + 0.3 × content-based")
print("  • Helps with cold-start problem")
print("  • Increases recommendation diversity")

print("\n2. DEEP LEARNING APPROACHES")
print("-" * 80)
print("Neural Collaborative Filtering (NCF):")
print("  • Replace dot product with neural network")
print("  • Learn non-linear user-item interactions")
print("  • Can incorporate side information (timestamps, user demographics)")
print("  • Expected improvement: 5-10% better RMSE")

print("\n3. TEMPORAL DYNAMICS")
print("-" * 80)
print("Account for time-changing preferences:")
print("  • Weight recent ratings more heavily")
print("  • Time-aware matrix factorization (TimeSVD++)")
print("  • Detect concept drift in user preferences")
print("  • Important for long-term users")

print("\n4. COLD-START PROBLEM")
print("-" * 80)
print("Strategies to handle new users/items:")
print("  • New users: Popularity-based recommendations initially")
print("  • New items: Content-based features (genre, actors, keywords)")
print("  • Active learning: Ask strategic questions to new users")
print("  • Transfer learning: Use data from similar domains")

print("\n5. DIVERSITY & SERENDIPITY")
print("-" * 80)
print("Improve recommendation quality beyond accuracy:")
print("  • Diversity: Recommend from different genres")
print("  • Serendipity: Include unexpected but relevant items")
print("  • Novelty: Don't just recommend popular items")
print("  • Re-ranking algorithms to balance accuracy vs diversity")

print("\n6. SCALABILITY FOR PRODUCTION")
print("-" * 80)
print("Optimize for real-world deployment:")
print("  • Approximate Nearest Neighbors (Annoy, FAISS) for faster search")
print("  • Online learning: Update model incrementally")
print("  • Distributed computing: Spark MLlib for large datasets")
print("  • Caching: Precompute recommendations for active users")
print("  • A/B testing: Validate improvements with real users")

print("\n7. EVALUATION METRICS")
print("-" * 80)
print("Beyond RMSE/MAE:")
print("  • Precision@K, Recall@K: Top-N recommendation quality")
print("  • NDCG (Normalized Discounted Cumulative Gain): Ranking quality")
print("  • Hit Rate: Percentage of relevant items in top-N")
print("  • User satisfaction surveys (qualitative)")

## 8. Save Final Comparison Report

In [ ]:
# Save comprehensive comparison table
all_results.to_csv('final_algorithm_comparison.csv', index=False)
print("Final comparison saved to: final_algorithm_comparison.csv")

# Create summary report
summary = f"""
MOVIE RECOMMENDATION SYSTEM - ALGORITHM COMPARISON REPORT
{'='*80}

Dataset: MovieLens Temporal Split
Training Set: {all_results.iloc[0]['test_samples'] * 4:,.0f} ratings (approx)
Test Set: {all_results.iloc[0]['test_samples']:,.0f} ratings

RESULTS SUMMARY:
{'-'*80}
{comparison_df.to_string(index=False)}
{'-'*80}

WINNER: {best_rmse_algo} with RMSE = {best_rmse:.4f}

RECOMMENDATIONS:
1. For production deployment: Use SVD for best accuracy
2. For interpretability: Use Item-Based CF
3. For small-scale systems: Use Item-Based or User-Based CF
4. For hybrid approach: Combine SVD + content-based filtering

NEXT STEPS:
- Implement hybrid recommender (CF + content-based)
- Address cold-start with content features
- Optimize for production deployment
- Add diversity/serendipity measures
"""

with open('algorithm_comparison_summary.txt', 'w') as f:
    f.write(summary)

print("\nSummary report saved to: algorithm_comparison_summary.txt")
print("\n" + summary)

## Summary

This notebook provided a comprehensive comparison of three collaborative filtering algorithms:

1. **Item-Based CF**: Stable, interpretable, good for cold-start users
2. **User-Based CF**: Intuitive, social-proof style, good for small systems
3. **SVD**: Best accuracy, fast predictions, production-ready

**Key Findings**:
- SVD typically achieves the best RMSE/MAE
- Item-Based CF offers best interpretability
- User-Based CF works well for social recommendations
- All three have trade-offs between accuracy, speed, and interpretability

**For Your Report**:
Use the visualizations and comparison tables generated here to support your analysis and recommendations.